In [14]:
import pandas as pd
import numpy as np
import sys, os
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Add project root to path
project_root = Path.cwd().parent  # parent of notebooks/
sys.path.append(str(project_root))

from src.data_preprocessing import Preprocessing


## init data

In [15]:

def init_data(df: pd.DataFrame):
    new_df = df.copy()

    processing = Preprocessing()
    processing.fit(new_df)
    new_df = processing.transform(new_df)

    X = new_df
    y = df['churn']

    # dataset을 train/test set으로 분리
    return train_test_split(
        X,
        y,
        test_size=0.25,
        stratify=y,
        random_state=42
    )

df = pd.read_csv("../data/bank_customer_churn_prediction.csv")
X_train, X_test, y_train, y_test = init_data(df)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

>>>>> final columns :  Index(['tenure', 'credit_score', 'estimated_salary', 'products_number',
       'balance', 'credit_card', 'active_member', 'gender_Female',
       'gender_Male', 'country_France', 'country_Germany', 'country_Spain',
       'age_group', 'age_x_active', 'products_x_active', 'age_x_products',
       'estimated_x_age', 'estimated_x_active'],
      dtype='object')


((7500, 18), (2500, 18), (7500,), (2500,))

## K-Fold init data

In [16]:

def init_data(df: pd.DataFrame):
    new_df = df.copy()

    processing = Preprocessing()
    processing.fit(new_df)
    new_df = processing.transform(new_df)

    X = new_df
    y = df['churn']

    # dataset을 train/test set으로 분리
    return train_test_split(
        X,
        y,
        test_size=0.25,
        stratify=y,
        random_state=42
    )

df = pd.read_csv("../data/bank_customer_churn_prediction.csv")
X_train, X_test, y_train, y_test = init_data(df)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

>>>>> final columns :  Index(['tenure', 'credit_score', 'estimated_salary', 'products_number',
       'balance', 'credit_card', 'active_member', 'gender_Female',
       'gender_Male', 'country_France', 'country_Germany', 'country_Spain',
       'age_group', 'age_x_active', 'products_x_active', 'age_x_products',
       'estimated_x_age', 'estimated_x_active'],
      dtype='object')


((7500, 18), (2500, 18), (7500,), (2500,))

In [17]:
def get_top_n_features(feature_names, importances, n=5):
    # 절대값 변환
    abs_imp = np.abs(importances)
    
    # DataFrame으로 정리
    df_imp = pd.DataFrame({
        "feature": feature_names,
        "importance": abs_imp
    })

    # 중요도 내림차순 정렬
    df_sorted = df_imp.sort_values("importance", ascending=False)

    # 상위 N개 feature 이름 리턴
    return df_sorted.head(n)["feature"].tolist()

### model 비교

In [11]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=8, random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        max_depth=-1,          # 제한 해제
        num_leaves=31,         # 필요에 따라 더 크게
        min_child_samples=10,  # 기본값보다 낮춰 분할 허용
        min_split_gain=0.0,    # 분할 이득 기준 완화
        learning_rate=0.05,
        random_state=42
    )
}

In [ ]:
results = []
probas = []

for name, model in models.items():
    model.fit(X_train, y_train)

    # 'age_x_active', 'products_x_active', 'age_x_products
    if name == "LogisticRegression":
        # ['active_member', 'gender_Male', 'country_France', 'country_Spain', 'age_group']
        feature_importances = get_top_n_features(X_train.columns, model.coef_[0])
        print(f"{name} FEATURE IMPORTANCES : \n{feature_importances}")
    else:
        # ['age', 'products_number', 'age_group']
        feature_importances = get_top_n_features(X_train.columns, model.feature_importances_)
        print(f"{name} FEATURE IMPORTANCES : \n{feature_importances}")

    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    auc = roc_auc_score(y_test, proba)

    results.append([name, accuracy, precision, recall, f1, auc])
    probas.append([name, pred, proba])

results

LogisticRegression FEATURE IMPORTANCES : 
['active_member', 'gender_Male', 'country_France', 'country_Spain', 'age_group']
RandomForest FEATURE IMPORTANCES : 
['age', 'products_number', 'age_group', 'balance', 'active_member']
XGBoost FEATURE IMPORTANCES : 
['age_group', 'products_number', 'active_member', 'age', 'country_Germany']
[LightGBM] [Info] Number of positive: 1528, number of negative: 5972
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000338 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 866
[LightGBM] [Info] Number of data points in the train set: 7500, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.203733 -> initscore=-1.363122
[LightGBM] [Info] Start training from score -1.363122
LightGBM FEATURE IMPORTANCES : 
['estimated_salary', 'credit_score', 'balance', 'age', 'tenure']


[['LogisticRegression',
  0.8144,
  0.6380368098159509,
  0.2043222003929273,
  0.30952380952380953,
  0.7872281849856771],
 ['RandomForest',
  0.8684,
  0.8383458646616542,
  0.4381139489194499,
  0.5754838709677419,
  0.8713947537987743],
 ['XGBoost',
  0.868,
  0.7640117994100295,
  0.5088408644400786,
  0.6108490566037735,
  0.8710671499152868],
 ['LightGBM',
  0.8656,
  0.7464387464387464,
  0.5147347740667977,
  0.6093023255813953,
  0.8663780726432007]]

In [18]:
# Interaction Feature 이전

columns = ["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
df_results = pd.DataFrame(results, columns=columns)
df_results

# 모두 높을수록 좋다.
# 1. Recall : 이탈 확률이 높은 고객의 정확도. 이탈 확률이 높은 고객을 미리 알고 대응할 수 있기때문에 이탈 문제에서 중요한 평가지표. **
# 2. Precision : churn이라고 예측한 고객 중 진짜 churn인 비율
# 3. F1-score : (Precision + Recall 균형)
# 4. ROC-AUC : 모델 종합 능력 평가

NameError: name 'results' is not defined

### Hyperparameter 적용 버전

In [19]:
from src.model_tuner import MultiModelTuner

models_with_params = {
    "LogisticRegression": (
        LogisticRegression(max_iter=3000, random_state=42),
        {
            "C": [0.001, 0.01, 0.1, 1, 3, 5, 10],   # 규제 강도
            "penalty": ["l1", "l2", "elasticnet"],
            "solver": ["liblinear", "saga"],
            "l1_ratio": [0, 0.3, 0.5, 0.7, 1]       # elasticnet 전용
        }
    ),

    "RandomForest": (
        RandomForestClassifier(random_state=42),
        {
            "n_estimators": [200, 300, 500],
            "max_depth": [4, 6, 8, 10],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
        }
    ),

    "XGBoost": (
        XGBClassifier(eval_metric="logloss", random_state=42),
        {
            "n_estimators": [200, 300, 500],
            "max_depth": [3, 4, 5, 6],
            "learning_rate": [0.01, 0.05, 0.1],
            "subsample": [0.6, 0.8, 1.0],
            "colsample_bytree": [0.6, 0.8, 1.0],
        }
    ),

    "LightGBM": (
        LGBMClassifier(random_state=42),
        {
            "n_estimators": [200, 300, 500],
            "max_depth": [-1, 4, 6, 8],
            "learning_rate": [0.01, 0.05, 0.1],
            "num_leaves": [15, 31, 63],
            "subsample": [0.6, 0.8, 1.0],
        }
    )
}

tuner = MultiModelTuner(models_with_params, n_iter=20)


In [20]:
best_models = tuner.tune(X_train, y_train)

▶ Tuning LogisticRegression ...


/Users/kim/Documents/lucyMacBookPro/SKN/팀프로젝트_2차/SKN21-2nd-1Team/grkim/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/kim/Documents/lucyMacBookPro/SKN/팀프로젝트_2차/SKN21-2nd-1Team/grkim/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/kim/Documents/lucyMacBookPro/SKN/팀프로젝트_2차/SKN21-2nd-1Team/grkim/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/Users/kim/Documents/lucyMacBookPro/SKN/팀프로젝트_2차/SKN21-2nd-1Team/grkim/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l

✔ LogisticRegression tuning completed. Best model found.
  → {'solver': 'saga', 'penalty': 'l2', 'l1_ratio': 0.5, 'C': 0.01}

▶ Tuning RandomForest ...
✔ RandomForest tuning completed. Best model found.
  → {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_depth': 10}

▶ Tuning XGBoost ...
✔ XGBoost tuning completed. Best model found.
  → {'subsample': 0.8, 'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.6}

▶ Tuning LightGBM ...
[LightGBM] [Info] Number of positive: 1018, number of negative: 3982
[LightGBM] [Info] Number of positive: 1019, number of negative: 3981
[LightGBM] [Info] Number of positive: 1018, number of negative: 3982
[LightGBM] [Info] Number of positive: 1019, number of negative: 3981
[LightGBM] [Info] Number of positive: 1019, number of negative: 3981
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001335 seconds.
You can set `force_col_wise=true` to remove the overhead.


In [ ]:
X_train.columns, best_models

(Index(['customer_id', 'credit_score', 'country', 'gender', 'age', 'tenure',
        'balance', 'products_number', 'credit_card', 'active_member',
        'estimated_salary'],
       dtype='object'),
 {'LogisticRegression': LogisticRegression(C=3, l1_ratio=0.3, max_iter=3000, penalty='l1',
                     random_state=42, solver='liblinear'),
  'RandomForest': RandomForestClassifier(max_depth=10, min_samples_leaf=4, min_samples_split=10,
                         n_estimators=300, random_state=42),
  'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
                colsample_bylevel=None, colsample_bynode=None,
                colsample_bytree=0.8, device=None, early_stopping_rounds=None,
                enable_categorical=False, eval_metric='logloss',
                feature_types=None, feature_weights=None, gamma=None,
                grow_policy=None, importance_type=None,
                interaction_constraints=None, learning_rate=0.01, max_bin=None,
     

In [ ]:
df_results = []

for name, model in best_models.items():
    if name == "LogisticRegression":
        feature_importances = get_top_n_features(X_train.columns, model.coef_[0])
        print(f"{name} FEATURE IMPORTANCES : \n{feature_importances}")
    else:
        feature_importances = get_top_n_features(X_train.columns, model.feature_importances_)
        print(f"{name} FEATURE IMPORTANCES : \n{feature_importances}")


    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    auc = roc_auc_score(y_test, proba)

    df_results.append([name, accuracy, precision, recall, f1, auc])

for result in df_results:
    print(result)

NameError: name 'X_test' is not defined

In [17]:
# Interaction Feature 이후 버전
import pandas as pd

columns = ["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
df_results = pd.DataFrame(df_results, columns=columns)
df_results

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,LogisticRegression,0.8248,0.762963,0.202358,0.319876,0.793786
1,RandomForest,0.8716,0.833333,0.461690,0.594185,0.873945
2,XGBoost,0.8740,0.825503,0.483301,0.609665,0.877965
3,LightGBM,0.8716,0.817568,0.475442,0.601242,0.877809


# Pipeline 적용 ==========================================

In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

import sys
from pathlib import Path
# Add project root to path
project_root = Path.cwd().parent  # parent of notebooks/
sys.path.append(str(project_root))

from src.data_preprocessing import PipelinePreprocessing

df = pd.read_csv("../data/bank_customer_churn_prediction.csv")

X = df.drop(columns=["churn"])
y = df["churn"]

pipeline_preprocessor = PipelinePreprocessing()
pre_pipeline = pipeline_preprocessor.create_pipeline(X)

final_pipeline = Pipeline([
    ("PipelinePreprocessing", pre_pipeline)
])

ValueError: Invalid parameter 'colsample_bytree' for estimator Pipeline(steps=[('PipelinePreprocessing',
                 Pipeline(steps=[('interaction_preprocessor',
                                  InteractionFeaturePreProcessing()),
                                 ('transformer',
                                  ColumnTransformer(remainder='passthrough',
                                                    transformers=[('id_dropper',
                                                                   'drop',
                                                                   ['customer_id']),
                                                                  ('standard_scaler',
                                                                   StandardScaler(),
                                                                   ['age',
                                                                    'tenure',
                                                                    'cre